# Homodimer Confidence Diagnostic Notebook
This notebook follows the PRD spec for a predicted AlphaFold Database homodimer. It recomputes the five confidence metrics and explains the results in beginner-friendly language.


## What this notebook does
- Loads a single AFDB homodimer prediction by accession ID
- Parses mmCIF atomic coordinates, PAE data, and pLDDT confidence data
- Detects the inter-chain interface using CB-CB distances (CA for glycine)
- Recomputes ipTM, ipSAE variants, pDockQ, pDockQ2, and LIS from first principles
- Shows visual summaries that make the score differences easy to understand
- Provides a diagnostic statement for users with no structural biology background


## 1. Setup and dependencies
This notebook uses only minimal scientific packages so it remains reproducible and easy to run.


In [ ]:
import json
import math
import shlex
from collections import OrderedDict
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'font.size': 12})
sns.set_style('whitegrid')

COLOR_CHAIN_A = '#009688'
COLOR_CHAIN_B = '#FF7043'
COLOR_INTERFACE = '#FFC107'
COLOR_PAE = 'viridis'


## 2. Helper functions and metric formulas
The following functions implement the IPSAE formulas exactly and preserve the asymmetry of the PAE matrix.


In [ ]:
def d0_length(L):
    if L <= 27:
        return 1.0
    return max(1.0, 1.24 * ((L - 15) ** (1/3)) - 1.8)

def ptm_score(pae, d0):
    return 1.0 / (1.0 + (pae / d0) ** 2)

def parse_mmCIF(cif_text):
    headers = []
    rows = []
    in_atom_loop = False
    for raw_line in cif_text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('loop_'):
            headers = []
            in_atom_loop = False
            continue
        if line.startswith('_atom_site.'):
            headers.append(line)
            in_atom_loop = True
            continue
        if in_atom_loop:
            try:
                parts = shlex.split(line)
            except ValueError:
                continue
            if len(parts) != len(headers):
                continue
            row = dict(zip(headers, parts))
            if row.get('_atom_site.group_PDB', '') != 'ATOM':
                continue
            alt_id = row.get('_atom_site.label_alt_id', '.')
            if alt_id not in ('', '.', 'A'):
                continue
            rows.append(row)

    chain_order = []
    chain_residues = {}
    for row in rows:
        chain_id = row['_atom_site.label_asym_id']
        if chain_id not in chain_order:
            chain_order.append(chain_id)
        resnum_token = row.get('_atom_site.auth_seq_id', row.get('_atom_site.label_seq_id'))
        try:
            resnum = int(resnum_token)
        except ValueError:
            resnum = int(row['_atom_site.label_seq_id'])
        atom_name = row['_atom_site.auth_atom_id']
        res_name = row['_atom_site.auth_comp_id']
        x = float(row['_atom_site.Cartn_x'])
        y = float(row['_atom_site.Cartn_y'])
        z = float(row['_atom_site.Cartn_z'])
        b_factor = float(row['_atom_site.B_iso_or_equiv'])
        key = (chain_id, resnum)
        if key not in chain_residues:
            chain_residues[key] = {'resname': res_name, 'atoms': {}}
        chain_residues[key]['atoms'][atom_name] = {'coord': np.array([x, y, z], dtype=float), 'b': b_factor}

    result = {'chain_ids': chain_order, 'chains': OrderedDict()}
    for chain_id in chain_order:
        residues = [(resnum, entry) for (cid, resnum), entry in chain_residues.items() if cid == chain_id]
        residues.sort(key=lambda x: x[0])
        residue_numbers = [resnum for resnum, _ in residues]
        residue_names = [entry['resname'] for _, entry in residues]
        ca_coords = []
        cb_coords = []
        ca_plddt = []
        for _, entry in residues:
            atoms = entry['atoms']
            if 'CA' not in atoms:
                raise ValueError(f'CA atom missing for residue in chain {chain_id}')
            ca_coords.append(atoms['CA']['coord'])
            ca_plddt.append(atoms['CA']['b'])
            if 'CB' in atoms:
                cb_coords.append(atoms['CB']['coord'])
            else:
                cb_coords.append(atoms['CA']['coord'])
        result['chains'][chain_id] = {
            'residue_numbers': residue_numbers,
            'residue_names': residue_names,
            'ca_coords': np.vstack(ca_coords),
            'cb_coords': np.vstack(cb_coords),
            'ca_plddt': np.array(ca_plddt, dtype=float),
        }
    return result

def parse_pae_json(pae_json):
    if not isinstance(pae_json, list) or len(pae_json) == 0:
        raise ValueError('Expected a list with PAE contents')
    entry = pae_json[0]
    pae_matrix = np.array(entry['predicted_aligned_error'], dtype=float)
    pae_chains = entry['chains']
    return pae_matrix, pae_chains

def parse_confidence_json(conf_json):
    scores = np.array(conf_json['confidenceScore'], dtype=float)
    confidence_by_chain = OrderedDict()
    index = 0
    for chain in conf_json['chains']:
        length = chain['sequenceEnd'] - chain['sequenceStart'] + 1
        confidence_by_chain[chain['label_asym_id']] = scores[index:index + length]
        index += length
    return confidence_by_chain

def align_structure_to_pae_order(structure, pae_chains):
    ordered = OrderedDict()
    for chain_info in pae_chains:
        label = chain_info['label_asym_id']
        if label not in structure['chains']:
            raise ValueError(f'Chain {label} found in PAE metadata but missing from mmCIF structure')
        ordered[label] = structure['chains'][label]
    structure['chain_ids'] = list(ordered.keys())
    structure['chains'] = ordered
    return structure

def fetch_afdb_prediction(accession_id):
    url = f'https://alphafold.ebi.ac.uk/api/prediction/{accession_id}'
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    entries = response.json()
    if not entries:
        raise ValueError(f'No prediction returned for {accession_id}')
    return entries[0]

def download_text(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.text

def load_homodimer_data(accession_id):
    entry = fetch_afdb_prediction(accession_id)
    cif_text = download_text(entry['cifUrl'])
    pae_json = requests.get(entry['paeDocUrl'], timeout=30).json()
    confidence_json = requests.get(entry['plddtDocUrl'], timeout=30).json()
    structure = parse_mmCIF(cif_text)
    pae_matrix, pae_chains = parse_pae_json(pae_json)
    structure = align_structure_to_pae_order(structure, pae_chains)
    confidence_by_chain = parse_confidence_json(confidence_json)
    return {
        'accession_id': accession_id,
        'entry': entry,
        'cif_text': cif_text,
        'structure': structure,
        'pae_matrix': pae_matrix,
        'pae_chains': pae_chains,
        'confidence_by_chain': confidence_by_chain,
    }


## 3. Load a homodimer prediction from AFDB
Change the accession ID below to analyse a different homodimer. The default is a valid AFDB accession for demonstration.


In [ ]:
accession_id = 'AF-0000000065889468'
print(f'Loading data for {accession_id}...')
prediction = load_homodimer_data(accession_id)
structure = prediction['structure']
pae_matrix = prediction['pae_matrix']
confidence_by_chain = prediction['confidence_by_chain']


## 4. Basic metadata and structure summary
This section shows the chains, lengths, and values that will be used for every score calculation.


In [ ]:
chain_ids = structure['chain_ids']
chain_lengths = {chain: len(structure['chains'][chain]['residue_numbers']) for chain in chain_ids}
total_residues = pae_matrix.shape[0]
print('AFDB accession:', prediction['accession_id'])
print('UniProt accession:', prediction['entry'].get('uniprotAccession', 'unknown'))
print('Chain order:', chain_ids)
print('Residues per chain:', chain_lengths)
print('PAE matrix shape:', pae_matrix.shape)
print('Total residues from PAE data:', total_residues)


## 5. Interface detection
We identify the inter-chain interface using CB-CB distances (CA for glycine). This is how pDockQ and pDockQ2 define contacts.


In [ ]:
def contact_matrix(cb_coords_1, cb_coords_2, cutoff=8.0):
    diff = cb_coords_1[:, None, :] - cb_coords_2[None, :, :]
    distances = np.linalg.norm(diff, axis=-1)
    return distances, distances <= cutoff

chain_a, chain_b = chain_ids[0], chain_ids[1]
cb_a = structure['chains'][chain_a]['cb_coords']
cb_b = structure['chains'][chain_b]['cb_coords']
dist_ab, contact_ab = contact_matrix(cb_a, cb_b, cutoff=8.0)
interface_a = contact_ab.any(axis=1)
interface_b = contact_ab.any(axis=0)
n_contacts = int(contact_ab.sum())
print(f'Inter-chain contact pairs (<=8 Å): {n_contacts}')
print(f'Interface residues in chain {chain_a}: {int(interface_a.sum())} / {len(interface_a)}')
print(f'Interface residues in chain {chain_b}: {int(interface_b.sum())} / {len(interface_b)}')


## 6. Score computation
This section recomputes ipTM, ipSAE variants, pDockQ, pDockQ2, and LIS from the original formulas.


In [ ]:
def iptm_direction(pae, chain1_idx, chain2_idx):
    block = pae[np.ix_(chain1_idx, chain2_idx)]
    d0 = d0_length(len(chain1_idx) + len(chain2_idx))
    row_mean = np.mean(ptm_score(block, d0), axis=1)
    return float(np.max(row_mean))

def ipsae_direction(pae, chain1_idx, chain2_idx, pae_cutoff=10.0, mode='d0res'):
    block = pae[np.ix_(chain1_idx, chain2_idx)]
    valid = block < pae_cutoff
    if mode == 'd0chn':
        d0_value = d0_length(len(chain1_idx) + len(chain2_idx))
    elif mode == 'd0dom':
        count1 = int(np.any(valid, axis=1).sum())
        count2 = int(np.any(valid, axis=0).sum())
        d0_value = d0_length(count1 + count2)
    scores = []
    for i, row in enumerate(block):
        row_valid = row[valid[i]]
        if row_valid.size == 0:
            scores.append(0.0)
            continue
        if mode == 'd0res':
            d0_value = d0_length(row_valid.size)
        scores.append(float(np.mean(ptm_score(row_valid, d0_value))))
    return float(np.max(scores))

def compute_interface_scores(pae, chain1_idx, chain2_idx, plddt1, plddt2, contact_mask):
    iptm = iptm_direction(pae, chain1_idx, chain2_idx)
    ipsae_d0res = ipsae_direction(pae, chain1_idx, chain2_idx, mode='d0res')
    ipsae_d0chn = ipsae_direction(pae, chain1_idx, chain2_idx, mode='d0chn')
    ipsae_d0dom = ipsae_direction(pae, chain1_idx, chain2_idx, mode='d0dom')
    contact_pairs = np.argwhere(contact_mask)
    n_contacts = len(contact_pairs)
    interface_rows = np.any(contact_mask, axis=1)
    interface_cols = np.any(contact_mask, axis=0)
    if n_contacts > 0:
        interface_plddt = np.concatenate([plddt1[interface_rows], plddt2[interface_cols]])
        mean_interface_plddt = float(np.mean(interface_plddt))
        x_pdockq = mean_interface_plddt * math.log10(max(n_contacts, 1))
        pdockq = 0.724 / (1 + math.exp(-0.052 * (x_pdockq - 152.611))) + 0.018
        contact_pae = pae[np.ix_(chain1_idx, chain2_idx)][contact_mask]
        mean_ptm = float(np.mean(ptm_score(contact_pae, 10.0)))
        x_pdockq2 = mean_interface_plddt * mean_ptm
        pdockq2 = 1.31 / (1 + math.exp(-0.075 * (x_pdockq2 - 84.733))) + 0.005
    else:
        mean_interface_plddt = 0.0
        pdockq = 0.0
        pdockq2 = 0.0
    lis_values = pae[np.ix_(chain1_idx, chain2_idx)][pae[np.ix_(chain1_idx, chain2_idx)] < 12.0]
    lis = float(np.mean((12.0 - lis_values) / 12.0)) if lis_values.size > 0 else 0.0
    return {
        'ipTM': iptm,
        'ipSAE_d0res': ipsae_d0res,
        'ipSAE_d0chn': ipsae_d0chn,
        'ipSAE_d0dom': ipsae_d0dom,
        'pDockQ': pdockq,
        'pDockQ2': pdockq2,
        'LIS': lis,
        'mean_interface_pLDDT': mean_interface_plddt,
        'n_contacts': n_contacts,
    }

chain_a_idx = np.arange(0, len(structure['chains'][chain_a]['residue_numbers']), dtype=int)
chain_b_idx = np.arange(len(structure['chains'][chain_a]['residue_numbers']), len(structure['chains'][chain_a]['residue_numbers']) + len(structure['chains'][chain_b]['residue_numbers']), dtype=int)
chain_scores = {
    chain_a: confidence_by_chain.get(chain_a, np.zeros(len(chain_a_idx), dtype=float)),
    chain_b: confidence_by_chain.get(chain_b, np.zeros(len(chain_b_idx), dtype=float)),
}
scores_a2b = compute_interface_scores(pae_matrix, chain_a_idx, chain_b_idx, chain_scores[chain_a], chain_scores[chain_b], contact_ab)
scores_b2a = compute_interface_scores(pae_matrix, chain_b_idx, chain_a_idx, chain_scores[chain_b], chain_scores[chain_a], contact_ab.T)
metrics = {
    'ipTM (A->B)': scores_a2b['ipTM'],
    'ipTM (B->A)': scores_b2a['ipTM'],
    'ipTM (symmetric max)': max(scores_a2b['ipTM'], scores_b2a['ipTM']),
    'ipSAE_d0res (A->B)': scores_a2b['ipSAE_d0res'],
    'ipSAE_d0res (B->A)': scores_b2a['ipSAE_d0res'],
    'ipSAE_d0res (symmetric max)': max(scores_a2b['ipSAE_d0res'], scores_b2a['ipSAE_d0res']),
    'ipSAE_d0chn (A->B)': scores_a2b['ipSAE_d0chn'],
    'ipSAE_d0chn (B->A)': scores_b2a['ipSAE_d0chn'],
    'ipSAE_d0dom (A->B)': scores_a2b['ipSAE_d0dom'],
    'ipSAE_d0dom (B->A)': scores_b2a['ipSAE_d0dom'],
    'pDockQ': scores_a2b['pDockQ'],
    'pDockQ2': scores_a2b['pDockQ2'],
    'LIS (A->B)': scores_a2b['LIS'],
    'LIS (B->A)': scores_b2a['LIS'],
    'LIS (symmetric average)': 0.5 * (scores_a2b['LIS'] + scores_b2a['LIS']),
    'mean_interface_pLDDT': scores_a2b['mean_interface_pLDDT'],
    'n_contacts': scores_a2b['n_contacts'],
}
print('Computed scores:')
for name, value in metrics.items():
    if isinstance(value, float):
        print(f'{name}: {value:.4f}')
    else:
        print(f'{name}: {value}')


## 7. Visualising the PAE matrix and interface
These plots make it easy to see where the two chains are confident, where the interface lies, and how the scores relate to the PAE geometry.


In [ ]:
def plot_pae_heatmap(pae, boundary):
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(pae, ax=ax, cmap=COLOR_PAE, cbar_kws={'label': 'PAE (Å)'}, square=True)
    ax.axhline(boundary, color='white', linewidth=2)
    ax.axvline(boundary, color='white', linewidth=2)
    ax.set_title('Full PAE matrix with chain boundaries')
    ax.set_xlabel('Residue index (concatenated chains)')
    ax.set_ylabel('Residue index (concatenated chains)')
    return fig, ax

def plot_contact_map(contact_mask):
    fig, ax = plt.subplots(figsize=(8, 6))
    plot = ax.imshow(contact_mask.astype(int), cmap='Blues', aspect='auto')
    ax.set_title('Inter-chain contact map (8 Å cutoff)')
    ax.set_xlabel(f'Chain {chain_b} residue index')
    ax.set_ylabel(f'Chain {chain_a} residue index')
    cbar = fig.colorbar(plot, ax=ax, pad=0.02)
    cbar.set_label('Contact (1 = contact)')
    return fig, ax

def plot_interface_coverage(interface_a, interface_b):
    fig, ax = plt.subplots(figsize=(10, 2.5))
    ax.bar(np.arange(len(interface_a)), interface_a.astype(int), color=COLOR_CHAIN_A, label=f'Chain {chain_a}')
    ax.bar(np.arange(len(interface_b)), interface_b.astype(int), bottom=0.0, color=COLOR_CHAIN_B, alpha=0.8, label=f'Chain {chain_b}')
    ax.set_title('Interface residue coverage')
    ax.set_xlabel('Residue index within each chain')
    ax.set_ylabel('Interface residue (1 = contact)')
    ax.set_ylim(0, 1.2)
    ax.legend(loc='upper right')
    return fig, ax

boundary = len(chain_a_idx)
plot_pae_heatmap(pae_matrix, boundary)
plt.show()
plot_contact_map(contact_ab)
plt.show()
plot_interface_coverage(interface_a, interface_b)
plt.tight_layout()
plt.show()


## 8. Optional 3D structure preview with MolViewSpec
If `molviewspec` is available, this cell will show an interactive structure viewer. Otherwise it will print an explanatory fallback message.


In [ ]:
try:
    import molviewspec
    if hasattr(molviewspec, 'MolViewSpec'):
        viewer = molviewspec.MolViewSpec.from_string(prediction['cif_text'], format='cif')
        display(viewer)
    else:
        print('MolViewSpec is installed, but the expected API is not available.')
except Exception as exc:
    print('MolViewSpec preview is unavailable in this environment.')
    print('Install molviewspec to enable 3D structure rendering.')
    print('Error:', exc)


## 9. Diagnostic summary
Use the computed scores and the visualisations above to understand whether the homodimer is globally confident, locally confident, or structurally plausible.
